# Capítulo 7 — Aprendizado Não Supervisionado

**Livro:** começa na página **314** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
rng = np.random.default_rng(42)
print("Pronto. pandas", pd.__version__)

## Calculando os Componentes Principais · PDF p. 318

#### 🧪 Exercício: PCA à mão × scikit-learn

Nos dados de vinhos padronizados, calcule os componentes principais pelos autovetores da matriz de covariância e compare a variância explicada com `PCA().explained_variance_ratio_`.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(load_wine().data)
autoval = np.sort(np.linalg.eigvalsh(np.cov(X, rowvar=False)))[::-1]
print((autoval / autoval.sum())[:4].round(3), PCA().fit(X).explained_variance_ratio_[:4].round(3))

## Escolhendo o Número de Grupos · PDF p. 328

#### 🧪 Exercício: K-médias e silhueta

Nos vinhos padronizados, rode k-médias para k = 2 a 8 e calcule a silhueta média de cada k. Qual k a silhueta sugere?

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
X = StandardScaler().fit_transform(load_wine().data)
sil = {k: round(silhouette_score(X, KMeans(k, n_init=10, random_state=0).fit_predict(X)), 3) for k in range(2, 9)}
print(sil, max(sil, key=sil.get))

## O Dendrograma · PDF p. 332

#### 🧪 Exercício: Dendrograma

Faça o agrupamento hierárquico (ligação de Ward) de 30 vinhos padronizados com `scipy.cluster.hierarchy` e desenhe o dendrograma.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram
X = StandardScaler().fit_transform(load_wine().data)[::6]
dendrogram(linkage(X, "ward")); plt.show()

## Misturas de Normais · PDF p. 338

#### 🧪 Exercício: Mistura de normais e BIC

Gere dados de 3 normais 2D e ajuste `GaussianMixture` com 1 a 6 componentes. Escolha o número de componentes pelo menor BIC.

In [ ]:
from sklearn.mixture import GaussianMixture
X = np.vstack([rng.normal(c, 0.8, (150, 2)) for c in ([0, 0], [4, 4], [0, 5])])
bic = {k: round(GaussianMixture(k, random_state=0).fit(X).bic(X), 1) for k in range(1, 7)}
print(bic, min(bic, key=bic.get))

## Escalonando as Variáveis · PDF p. 344

#### 🧪 Exercício: Variável dominante

Rode k-médias (k = 3) nos vinhos **sem** padronizar e **padronizados**. Compare os grupos com as classes reais usando `adjusted_rand_score`. Qual variável domina sem padronização?

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
d = load_wine(as_frame=True)
for X in (d.data.to_numpy(), StandardScaler().fit_transform(d.data)):
    print(round(adjusted_rand_score(d.target, KMeans(3, n_init=10, random_state=0).fit_predict(X)), 3))
print(d.data.var().sort_values(ascending=False).head(2))  # proline tem variância enorme